# BanglaHalluEval: SOMADHAN Reasoning Hallucination Pipeline
#
# Input CSV columns: question, answer, reasoning_len
#   - question      : the Bengali math problem
#   - answer        : full CoT chain + #### + final answer  (split at ####)
#   - reasoning_len : character length of the answer field
#
# Workflow:
# 1. Install dependencies
# 2. Mount Drive
# 3. Configure OpenAI API key
# 4. Upload SOMADHAN CSV
# 5. Select first 1000 rows (length-sorted input expected)
# 6. Assign 200 items per hallucination type (round robin)
# 7. Generate hallucinated reasoning chains -> save CSV
# 8. Quality check
# 9. Download results

In [1]:
# @title 1. Install Dependencies
!pip -q install openai pandas

In [ ]:
# @title 2. Mount Drive
from google.colab import drive
from pathlib import Path
 
drive.mount('/content/drive')
 
BASE_DIR = '/content/drive/MyDrive/BanglaHalluEval'
Path(BASE_DIR).mkdir(parents=True, exist_ok=True)
print('Saving outputs to:', BASE_DIR)

In [ ]:
# @title 3. Configuration & API Setup
import csv
import hashlib
import json
import random
import re
import time
from pathlib import Path
 
import pandas as pd
from google.colab import files, userdata
from openai import OpenAI
 
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    OPENAI_API_KEY = ''  # paste key here if not using Colab Secrets
 
if not OPENAI_API_KEY:
    print('ERROR: No API key found. Add it to Colab Secrets or paste directly above.')
else:
    print('API key configured.')
 
MODEL_NAME       = 'gpt-5.4'  
RANDOM_SEED      = 42
CHECKPOINT_EVERY = 1
 
client = OpenAI(api_key=OPENAI_API_KEY)
 
DRIVE_OUT = Path(BASE_DIR) / 'Results' / 'reasoning_hallucination_results'
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
 
OUTPUT_1000         = DRIVE_OUT / 'somadhan_1000.csv'
OUTPUT_TYPED        = DRIVE_OUT / 'somadhan_1000_typed.csv'
OUTPUT_HALLUCINATED = DRIVE_OUT / 'somadhan_1000_hallucinated.csv'
LOG_PATH            = DRIVE_OUT / 'somadhan_1000_hallucinated.log'

In [ ]:
# @title 4. Upload Dataset (SOMADHAN_SORTED.csv)
print("Please upload 'SOMADHAN_SORTED.csv'")
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f'Uploaded: {INPUT_FILENAME}')

In [ ]:
# @title 5. Utility Functions
 
def load_rows(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))
 
def write_rows(path, rows, fieldnames):
    with open(path, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
 
def write_df(df, path):
    df.to_csv(path, index=False, encoding='utf-8-sig')
 
def append_log(path, text):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(text)
 
def split_chain_answer(text):
    """Split 'answer' column at #### into (CoT chain, final answer)."""
    cleaned = (text or '').strip()
    if '####' in cleaned:
        chain, answer = cleaned.rsplit('####', 1)
        return chain.strip(), answer.strip()
    return cleaned, ''
 
def make_question_id(question):
    cleaned = (question or '').strip()
    return hashlib.sha1(cleaned.encode('utf-8')).hexdigest()
 
def assign_types_round_robin(df, type_order, seed=None):
    """
    Assign hallucination types using round-robin across the sorted dataframe.
    Row i gets type_order[i % len(type_order)].
    Requires len(df) to be exactly divisible by len(type_order).
    """
    n_types = len(type_order)
    if len(df) % n_types != 0:
        raise ValueError(
            f"Dataset length {len(df)} is not divisible by number of types {n_types}. "
            "Trim or pad the dataset first."
        )
    df_out = df.copy().reset_index(drop=True)
    df_out['hallucination_type'] = [type_order[i % n_types] for i in range(len(df_out))]
    return df_out
 
def parse_json_response(text):
    """Strip markdown fences, repair unquoted Bengali numerals, parse JSON.
    Returns (dict|None, error|None)."""
    cleaned = (text or '').strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.split('\n', 1)[-1].strip()
        if cleaned.endswith('```'):
            cleaned = cleaned.rsplit('```', 1)[0].strip()
    cleaned = re.sub(
        r'("hallucinated_answer"\s*:\s*)([^",\}\]\n]+?)(\s*[\},\n\]])',
        lambda m: m.group(1) + '"' + m.group(2).strip() + '"' + m.group(3),
        cleaned
    )
    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed.get('hallucinated_chain'), list):
            parsed['hallucinated_chain'] = ' '.join(parsed['hallucinated_chain'])
        return parsed, None
    except Exception as e:
        return None, str(e)
 
def build_prompt(template, problem, chain, answer):
    return (template
            .replace('<PROBLEM>', problem)
            .replace('<CHAIN>', chain)
            .replace('<ANSWER>', answer))

In [ ]:
# @title 6. Select First 1000 Rows & Assign 5 Hallucination Types (Round Robin)
 
df = pd.read_csv(INPUT_FILENAME)
 
# Dataset must be pre-sorted by reasoning_len ascending before upload.
# 1000 rows / 5 types = exactly 200 per type.
df_1000 = df.head(1000).sort_values('reasoning_len').reset_index(drop=True)
write_df(df_1000, OUTPUT_1000)
 
TYPE_ORDER = [
    'arithmetic_slip',
    'formula_misapplication',
    'invalid_deduction',
    'hallucinated_intermediate_fact',
    'semantic_drift',
]
 
df_typed = assign_types_round_robin(df_1000, TYPE_ORDER)
write_df(df_typed, OUTPUT_TYPED)
 
print('Saved:', OUTPUT_1000)
print('Saved:', OUTPUT_TYPED)
print('\nType distribution:')
print(df_typed['hallucination_type'].value_counts())
 
# Sanity check: each type should have equal count and uniform length spread
print('\nLength stats per type:')
print(df_typed.groupby('hallucination_type')['reasoning_len'].describe().round(1))

In [ ]:
# @title 7. Define Prompts (5 hallucination types)
 
prompt_1 = (
    "I want you to act as a hallucinated reasoning chain generator. The output must be in BANGLA. "
    "Given a math problem, a correct step-by-step reasoning chain, and the correct final answer, "
    "your objective is to produce a hallucinated reasoning chain that sounds plausible but is mathematically incorrect. "
    "You MUST use the following hallucination method Arithmetic Slip: keep the logic structure completely intact but insert one subtle "
    "numerical error in a single calculation step. The wrong number should be close to the correct one "
    "(roughly 5% to 20% off). All subsequent steps must use the wrong number consistently, and the entire chain "
    "must remain in Bengali. Preserve the <<expr=result>> annotation format — in the modified step both expr and "
    "result must reflect the wrong value. Return ONLY a JSON object with keys: hallucinated_chain, error_step (integer), "
    "error_type, hallucinated_answer.\n"
    "Example —\n"
    "#Problem#: \"জিনাত ছয়জন কর্মচারী নিয়োগ করে। তাদের মধ্যে চারজন গুদাম কর্মী যারা ৳১৫/ঘণ্টা আয় করে এবং "
    "বাকি দুইজন ম্যানেজার যারা ৳২০/ঘন্টা করে। প্রত্যেকে যদি মাসে ২৫ দিন এবং দিনে ৮ ঘন্টা কাজ করে তাহলে "
    "এক মাসের মজুরি ও করের জন্য জিনাতের মোট কত দেনা আছে?\"\n"
    "#Correct Reasoning Chain#: \"প্রথমে প্রতিটি কর্মী প্রতি মাসে কত ঘন্টা কাজ করে তা বের করুন: "
    "২৫ দিন * ৮ ঘন্টা/দিন = <<২৫*৮=২০০>>২০০ ঘন্টা\"\n"
    "#Hallucinated Reasoning Chain#: \"প্রথমে প্রতিটি কর্মী প্রতি মাসে কত ঘন্টা কাজ করে তা বের করুন: "
    "২৫ দিন * ৮ ঘন্টা/দিন = <<২৫*৮=১৮০>>১৮০ ঘন্টা\"\n"
    "Note: Only the <<expr=result>> value and inline number changed. All later steps use ১৮০ instead of ২০০. "
    "You should try your best to make the hallucination subtle and believable.\n"
    "#Problem#: <PROBLEM>\n"
    "#Correct Reasoning Chain#: <CHAIN>\n"
    "#Correct Answer#: <ANSWER>\n"
    "#Hallucinated Chain#: Generate"
 )
 
prompt_2 = (
    "I want you to act as a hallucinated reasoning chain generator. The output must be in BANGLA. "
    "Given a math problem, a correct step-by-step reasoning chain, and the correct final answer, "
    "your objective is to produce a hallucinated reasoning chain that sounds plausible but is mathematically incorrect. "
    "You MUST use the following hallucination method Formula Misapplication: use a related but wrong operation in one step "
    "(e.g. divide instead of multiply, or add instead of multiply). The Bengali narrative in that step must "
    "justify the wrong operation so it still sounds natural. All subsequent steps must be consistent with the "
    "wrong result, and the entire chain must remain in Bengali. Preserve the <<expr=result>> annotation — in the "
    "modified step both expr and result must reflect the wrong operation and value. Return ONLY a JSON object "
    "with keys: hallucinated_chain, error_step (integer), error_type, hallucinated_answer.\n"
    "Example —\n"
    "#Problem#: \"জিনাত ছয়জন কর্মচারী নিয়োগ করে। প্রত্যেকে মাসে ২৫ দিন ৮ ঘন্টা কাজ করে। "
    "গুদাম কর্মীর হার ৳৫/ঘন্টা।\"\n"
    "#Correct Reasoning Chain#: \"তারপরে একজন গুদাম কর্মী প্রতি মাসে কত উপার্জন করে তা গণনা করুন: "
    "২০০ ঘন্টা * ৳৫/ঘন্টা = ৳<<২০০*৫=১০০০>>১০০০\"\n"
    "#Hallucinated Reasoning Chain#: \"মোট আয় বের করতে ঘন্টার সংখ্যাকে ঘন্টার হার দিয়ে যোগ করুন: "
    "২০০ + ৫ = ৳<<২০০+৫=২০৫>>২০৫\"\n"
    "Note: The Bengali text says 'যোগ করুন' (add) to justify the wrong operation. "
    "All later steps use ২০৫ as the per-worker earning. "
    "You should try your best to make the hallucination subtle and believable. "
    "Important: the wrong result must stay within the same order of magnitude as the correct intermediate value — "
    "do not choose an operation that produces a value more than 5x away from the correct intermediate value.\n"
    "#Problem#: <PROBLEM>\n"
    "#Correct Reasoning Chain#: <CHAIN>\n"
    "#Correct Answer#: <ANSWER>\n"
    "#Hallucinated Chain#: Generate"
 )
 
prompt_3 = (
    "I want you to act as a hallucinated reasoning chain generator. The output must be in BANGLA. "
    "Given a math problem, a correct step-by-step reasoning chain, and the correct final answer, "
    "your objective is to produce a hallucinated reasoning chain that sounds plausible but is mathematically incorrect. "
    "You MUST use the following hallucination method Invalid Deduction: inject one logically unsupported conclusion in a single step — "
    "a transition that does not actually follow from the previous step but sounds natural in Bengali. "
    "All subsequent steps must follow from this invalid conclusion, and the entire chain must remain in Bengali. "
    "Preserve the <<expr=result>> annotation format. Return ONLY a JSON object with keys: "
    "hallucinated_chain, error_step (integer), error_type, hallucinated_answer.\n"
    "Example —\n"
    "#Problem#: \"জিনাত ছয়জন কর্মচারী নিয়োগ করে। মোট মজুরি ৳২০,০০০।\"\n"
    "#Correct Reasoning Chain#: \"এখন ফিকা ট্যাক্স কত তা জানতে মোট মজুরি বিলকে ১০% দিয়ে গুণ করুন: "
    "৳২০,০০০ * .১ = ৳<<২০০০০*.১=২০০০>>২,০০০\"\n"
    "#Hallucinated Reasoning Chain#: \"যেহেতু কর্মচারীর সংখ্যা জোড় (৬ জন), তাই কর গণনার ক্ষেত্রে "
    "প্রতিটি জুটির জন্য আলাদাভাবে হিসাব করতে হবে। প্রতি জুটির মজুরি = ৳২০,০০০ ÷ ৩ = <<২০০০০/৩=৬৬৬৭>>৳৬,৬৬৭।\"\n"
    "Note: Dividing by number of pairs is not stated anywhere in the problem — it is a hallucinated logical step. "
    "All later steps use ৳৬,৬৬৭ as the base. "
    "You should try your best to make the hallucination subtle and believable.\n"
    "#Problem#: <PROBLEM>\n"
    "#Correct Reasoning Chain#: <CHAIN>\n"
    "#Correct Answer#: <ANSWER>\n"
    "#Hallucinated Chain#: Generate"
 )
 
prompt_4 = (
    "I want you to act as a hallucinated reasoning chain generator. The output must be in BANGLA. "
    "Given a math problem, a correct step-by-step reasoning chain, and the correct final answer, "
    "your objective is to produce a hallucinated reasoning chain that sounds plausible but is mathematically incorrect. "
    "You MUST use the following hallucination method Hallucinated Intermediate Fact : introduce one fabricated assumption not present anywhere in "
    "the problem as if it naturally follows from the problem context. All subsequent steps must proceed consistently "
    "from this fabricated fact, and the entire chain must remain in Bengali. Preserve the <<expr=result>> annotation "
    "format. Return ONLY a JSON object with keys: hallucinated_chain, error_step (integer), error_type, hallucinated_answer.\n"
    "Example —\n"
    "#Problem#: \"জিনাত ছয়জন কর্মচারী নিয়োগ করে। প্রত্যেকে মাসে ২৫ দিন ৮ ঘন্টা কাজ করে।\"\n"
    "#Correct Reasoning Chain#: \"প্রথমে প্রতিটি কর্মী প্রতি মাসে কত ঘন্টা কাজ করে তা বের করুন: "
    "২৫ দিন * ৮ ঘন্টা/দিন = <<২৫*৮=২০০>>২০০ ঘন্টা\"\n"
    "#Hallucinated Reasoning Chain#: \"প্রশ্নটি ইঙ্গিত করে যে কর্মীরা প্রতি মাসে ২ দিন ছুটি ভোগ করেন, "
    "তাই কার্যকর কাজের দিন = ২৫ - ২ = <<২৫-২=২৩>>২৩ দিন। সুতরাং মোট ঘন্টা = ২৩ * ৮ = <<২৩*৮=১৮৪>>১৮৪ ঘন্টা।\"\n"
    "Note: '2 days off' appears nowhere in the problem — entirely fabricated but sounds plausible. "
    "All later steps use ১৮৪ working hours instead of ২০০. "
    "You should try your best to make the hallucination subtle and believable.\n"
    "#Problem#: <PROBLEM>\n"
    "#Correct Reasoning Chain#: <CHAIN>\n"
    "#Correct Answer#: <ANSWER>\n"
    "#Hallucinated Chain#: Generate"
 )
 
prompt_5 = (
    "I want you to act as a hallucinated reasoning chain generator. The output must be in BANGLA. "
    "Given a math problem, a correct step-by-step reasoning chain, and the correct final answer, "
    "your objective is to produce a hallucinated reasoning chain that sounds plausible but is mathematically incorrect. "
    "You MUST use the following hallucination method Semantic Drift: slightly reinterpret what the problem is asking at some point "
    "mid-reasoning — for example, the problem asks for total cost (wages + tax) but the reasoning drifts to treating wages alone as the final answer, skipping a required step. The drift must feel like a plausible reading of the problem, not an obvious mistake. The entire chain must remain in Bengali. Preserve the <<expr=result>> annotation format." 
    "Return ONLY a JSON object with keys: hallucinated_chain, error_step (integer), error_type, hallucinated_answer.\n"
    "Example —\n"
    "#Problem#: \"জিনাতকে তার কর্মীদের বেতনের ১০% ফিকা ট্যাক্সে দিতে হবে। মোট মজুরি ও করের জন্য মোট কত দেনা?\"\n"
    "#Correct Reasoning Chain#: \"এখন মোট মজুরি বিল যোগ করে মোট ট্যাক্সের পরিমাণ খুঁজে বের করুন: "
    "৳২,০০০ + ৳২০,০০০ = ৳<<২০০০+২০০০০=২২০০০>>২২,০০০\"\n"
    "#Hallucinated Reasoning Chain#: \"যেহেতু প্রশ্নটি মূলত মজুরির মোট পরিমাণ জিজ্ঞেস করছে, তাই মোট মজুরি "
    "৳২০,০০০ই চূড়ান্ত উত্তর। ট্যাক্সের পরিমাণ নিয়োগকর্তার নিজস্ব ব্যয় হিসেবে আলাদা ধরা হয়।\"\n"
    "Note: The reasoning reinterprets 'wages + tax' as 'wages only', skipping the tax step entirely. "
    "hallucinated_answer becomes ২০০০০ instead of ২২০০০. "
    "You should try your best to make the hallucination subtle and believable.\n"
    "#Problem#: <PROBLEM>\n"
    "#Correct Reasoning Chain#: <CHAIN>\n"
    "#Correct Answer#: <ANSWER>\n"
    "#Hallucinated Chain#: Generate"
 )
 
PROMPT_MAP = {
    'arithmetic_slip':               prompt_1,
    'formula_misapplication':        prompt_2,
    'invalid_deduction':             prompt_3,
    'hallucinated_intermediate_fact': prompt_4,
    'semantic_drift':                prompt_5,
}

In [ ]:
# @title 8. Run Generation Loop
 
rows_df = pd.read_csv(OUTPUT_TYPED)
output_rows = []
completed_source_ids = set()
 
FIELDNAMES = [
    'id',
    'source_id',
    'question_id',
    'hallucination_type',
    'question',
    'answer',
    'reasoning_len',
    'hallucinated_chain',
    'error_step',
    'error_type',
    'hallucinated_answer',
    'raw_response',
    'parse_error',
 ]
 
if OUTPUT_HALLUCINATED.exists():
    output_rows = load_rows(str(OUTPUT_HALLUCINATED))
    completed_source_ids = {r.get('source_id') for r in output_rows if r.get('source_id')}
    print(f'Resuming: {len(completed_source_ids)} items already done.')
 
append_log(LOG_PATH, 'Starting reasoning hallucination generation.\n')
append_log(LOG_PATH, f'Total items: {len(rows_df)}\n')
 
for idx, row in rows_df.iterrows():
    source_id = str(row['id']) if 'id' in rows_df.columns else str(idx + 1)
 
    if source_id in completed_source_ids:
        continue
 
    question      = str(row.get('question', '')).strip()
    full_answer   = str(row.get('answer', '')).strip()
    reasoning_len = row.get('reasoning_len', '')
    h_type        = str(row.get('hallucination_type', '')).strip()
 
    if h_type not in PROMPT_MAP:
        append_log(LOG_PATH, f'[{idx+1}] Unknown hallucination_type={h_type} for source_id={source_id}\n')
        continue
 
    chain, correct_answer = split_chain_answer(full_answer)
    prompt = build_prompt(PROMPT_MAP[h_type], question, chain, correct_answer)
    question_id = make_question_id(question)
 
    raw_response = ''
    parsed       = None
    parse_error  = None
 
    try:
        response = client.responses.create(
            model=MODEL_NAME,
            input=[{'role': 'user', 'content': prompt}],
            max_output_tokens=900,
            temperature=0.7,
        )
        raw_response = response.output_text
        parsed, parse_error = parse_json_response(raw_response)
 
    except Exception as e:
        parse_error = f'API error: {e}'
        append_log(LOG_PATH, f'[{idx+1}] API error for source_id={source_id}: {e}\n')
 
    output_rows.append({
        'id':                  f"{source_id}::{h_type}",
        'source_id':           source_id,
        'question_id':         question_id,
        'hallucination_type':  h_type,
        'question':            question,
        'answer':              full_answer,
        'reasoning_len':       reasoning_len,
        'hallucinated_chain':  (parsed or {}).get('hallucinated_chain', ''),
        'error_step':          (parsed or {}).get('error_step', ''),
        'error_type':          (parsed or {}).get('error_type', ''),
        'hallucinated_answer': (parsed or {}).get('hallucinated_answer', ''),
        'raw_response':        raw_response,
        'parse_error':         parse_error or '',
    })
 
    if (len(output_rows) % CHECKPOINT_EVERY) == 0:
        write_rows(str(OUTPUT_HALLUCINATED), output_rows, FIELDNAMES)
        append_log(LOG_PATH,
            f'[{idx+1}/{len(rows_df)}] source_id={source_id} | type={h_type} | saved {len(output_rows)} rows\n')
        print(f'[{idx+1}/{len(rows_df)}] {h_type} | saved {len(output_rows)} rows')
 
    time.sleep(0.1)
 
write_rows(str(OUTPUT_HALLUCINATED), output_rows, FIELDNAMES)
print(f'\nDone. Total rows written: {len(output_rows)}')
print(f'Output: {OUTPUT_HALLUCINATED}')
print(f'Log:    {LOG_PATH}')

In [ ]:
# @title 9. Quality Check
 
df_out = pd.read_csv(OUTPUT_HALLUCINATED)
 
print('=== Total rows:', len(df_out))
print('\n--- Type distribution ---')
print(df_out['hallucination_type'].value_counts())
 
parse_errors = df_out[df_out['parse_error'].astype(str).str.len() > 0]
print(f'\n--- Parse errors: {len(parse_errors)} rows ---')
if len(parse_errors) > 0:
    print(parse_errors[['source_id', 'hallucination_type', 'parse_error']].head(10))
 
empty_chains = df_out[df_out['hallucinated_chain'].astype(str).str.len() == 0]
print(f'\n--- Empty hallucinated_chain: {len(empty_chains)} rows ---')
 
print('\n--- Sample output (first row) ---')
sample = df_out.iloc[0]
print('Type:             ', sample['hallucination_type'])
print('Error step:       ', sample['error_step'])
print('Error type:       ', sample['error_type'])
print('Hallucinated ans: ', sample['hallucinated_answer'])
print('Chain preview:    ', str(sample['hallucinated_chain'])[:300])

In [ ]:
# @title 10. Download Results
files.download(str(OUTPUT_1000))
files.download(str(OUTPUT_TYPED))
files.download(str(OUTPUT_HALLUCINATED))